# Benchmark Flat File Databases in R

This notebook benchmarks different flat file database engines:
- **DuckDB** (in-memory and persistent)
- **SQLite** (via RSQLite)
- **Parquet** (via arrow, queried with DuckDB)
- **CSV** (via data.table's fread)

We generate a synthetic dataset of 1 million rows × 8 columns, persist to each format, and run microbenchmarks (100 iterations) for various query types:
- Full table scan (`SELECT *`)
- Filtered queries (`WHERE` clauses)
- Aggregations (`GROUP BY` with `SUM`, `COUNT`, `AVG`)
- Summary statistics

Results include timing, file sizes, and memory profiling.

## 1. Setup and Dependencies

In [ ]:
# Install packages if needed (uncomment to install)
# install.packages(c("duckdb", "RSQLite", "arrow", "data.table", "DBI", 
#                    "microbenchmark", "ggplot2", "dplyr", "tidyr", "pryr", "scales"))

source("~/renv_start.R")

# Load libraries
library(duckdb)
library(RSQLite)
library(arrow)
library(data.table)
library(DBI)
library(microbenchmark)
library(ggplot2)
library(dplyr)
library(tidyr)
library(scales)
library(pryr)

# Set seed for reproducibility
set.seed(42)

# Configuration
N_ROWS <- 1000000L  # 1 million rows
N_BENCHMARK_RUNS <- 100L

cat("Configuration:\n")
cat(sprintf("  Rows: %s\n", format(N_ROWS, big.mark = ",")))
cat(sprintf("  Benchmark runs: %d\n", N_BENCHMARK_RUNS))

## 2. Generate Synthetic Dataset

Create a 1 million row dataset with 8 columns:
- `id`: Unique integer ID
- `category`: Categorical variable (A-J)
- `region`: Geographic region (10 regions)
- `value_int`: Random integer (1-10000)
- `value_float`: Random float (0-1000)
- `quantity`: Random integer (1-100)
- `date`: Random date in 2024
- `description`: Random text string

In [ ]:
# Generate synthetic data
generate_data <- function(n_rows) {
  # Categories and regions for variety
  categories <- LETTERS[1:10]  # A through J
  regions <- c("North", "South", "East", "West", "Central", 
               "Northeast", "Southeast", "Northwest", "Southwest", "Midwest")
  
  # Random words for description
  words <- c("alpha", "beta", "gamma", "delta", "epsilon", 
             "product", "service", "item", "unit", "widget",
             "premium", "standard", "basic", "pro", "lite")
  
  # Generate data
  df <- data.frame(
    id = 1:n_rows,
    category = sample(categories, n_rows, replace = TRUE),
    region = sample(regions, n_rows, replace = TRUE),
    value_int = sample(1:10000, n_rows, replace = TRUE),
    value_float = runif(n_rows, 0, 1000),
    quantity = sample(1:100, n_rows, replace = TRUE),
    date = as.Date("2024-01-01") + sample(0:364, n_rows, replace = TRUE),
    description = paste(
      sample(words, n_rows, replace = TRUE),
      sample(words, n_rows, replace = TRUE),
      sample(1000:9999, n_rows, replace = TRUE),
      sep = "_"
    ),
    stringsAsFactors = FALSE
  )
  
  return(df)
}

# Generate the data
cat("Generating synthetic data...\n")
start_time <- Sys.time()
df <- generate_data(N_ROWS)
generation_time <- difftime(Sys.time(), start_time, units = "secs")

cat(sprintf("Generated %s rows in %.2f seconds\n", 
            format(nrow(df), big.mark = ","), 
            as.numeric(generation_time)))

# Show structure and sample
cat("\nDataset structure:\n")
str(df)

cat("\nFirst 5 rows:\n")
head(df, 5)

# Memory footprint
cat(sprintf("\nDataFrame memory size: %s\n", 
            format(object.size(df), units = "MB")))

## 3. Persist Data to Each Format

Write the dataset to:
- DuckDB persistent database
- SQLite database  
- Parquet file
- CSV file

Also measure write times and file sizes.

In [ ]:
# Create output directory
output_dir <- "benchmark_data"
if (!dir.exists(output_dir)) {
  dir.create(output_dir)
}

# File paths
duckdb_path <- file.path(output_dir, "benchmark.duckdb")
sqlite_path <- file.path(output_dir, "benchmark.sqlite")
parquet_path <- file.path(output_dir, "benchmark.parquet")
csv_path <- file.path(output_dir, "benchmark.csv")

# Remove existing files
for (f in c(duckdb_path, sqlite_path, parquet_path, csv_path)) {
  if (file.exists(f)) file.remove(f)
}

# Track write times and sizes
write_stats <- list()

# 1. DuckDB
cat("Writing to DuckDB...\n")
start_time <- Sys.time()
con_duck <- dbConnect(duckdb(), dbdir = duckdb_path)
dbWriteTable(con_duck, "benchmark_data", df, overwrite = TRUE)
dbDisconnect(con_duck, shutdown = TRUE)
write_stats$duckdb <- list(
  time = as.numeric(difftime(Sys.time(), start_time, units = "secs")),
  size = file.info(duckdb_path)$size
)
cat(sprintf("  Time: %.2f sec, Size: %s\n", 
            write_stats$duckdb$time, 
            format(structure(write_stats$duckdb$size, class = "object_size"), units = "MB")))

# 2. SQLite
cat("Writing to SQLite...\n")
start_time <- Sys.time()
con_sqlite <- dbConnect(RSQLite::SQLite(), sqlite_path)
dbWriteTable(con_sqlite, "benchmark_data", df, overwrite = TRUE)
dbDisconnect(con_sqlite)
write_stats$sqlite <- list(
  time = as.numeric(difftime(Sys.time(), start_time, units = "secs")),
  size = file.info(sqlite_path)$size
)
cat(sprintf("  Time: %.2f sec, Size: %s\n", 
            write_stats$sqlite$time, 
            format(structure(write_stats$sqlite$size, class = "object_size"), units = "MB")))

# 3. Parquet
cat("Writing to Parquet...\n")
start_time <- Sys.time()
write_parquet(df, parquet_path)
write_stats$parquet <- list(
  time = as.numeric(difftime(Sys.time(), start_time, units = "secs")),
  size = file.info(parquet_path)$size
)
cat(sprintf("  Time: %.2f sec, Size: %s\n", 
            write_stats$parquet$time, 
            format(structure(write_stats$parquet$size, class = "object_size"), units = "MB")))

# 4. CSV
cat("Writing to CSV...\n")
start_time <- Sys.time()
fwrite(as.data.table(df), csv_path)
write_stats$csv <- list(
  time = as.numeric(difftime(Sys.time(), start_time, units = "secs")),
  size = file.info(csv_path)$size
)
cat(sprintf("  Time: %.2f sec, Size: %s\n", 
            write_stats$csv$time, 
            format(structure(write_stats$csv$size, class = "object_size"), units = "MB")))

# Summary
cat("\n=== Write Performance Summary ===\n")
write_summary <- data.frame(
  Engine = c("DuckDB", "SQLite", "Parquet", "CSV"),
  Write_Time_sec = c(write_stats$duckdb$time, write_stats$sqlite$time, 
                     write_stats$parquet$time, write_stats$csv$time),
  File_Size_MB = c(write_stats$duckdb$size, write_stats$sqlite$size,
                   write_stats$parquet$size, write_stats$csv$size) / (1024^2)
)
print(write_summary)

## 4. Define Benchmark Query Functions

Define query functions for each engine covering:
1. **Full Scan**: `SELECT * LIMIT 10000`
2. **Filter Numeric**: `WHERE value_int > 5000`
3. **Filter String**: `WHERE category = 'A'`
4. **Group By Sum**: `GROUP BY category` with `SUM(value_float)`
5. **Group By Multiple**: `GROUP BY category, region` with multiple aggregates
6. **Summary Stats**: Mean, min, max across numeric columns

In [ ]:
# ============================================================
# DuckDB Query Functions
# ============================================================

duckdb_queries <- list(
  full_scan = function(con) {
    dbGetQuery(con, "SELECT * FROM benchmark_data LIMIT 10000")
  },
  
  filter_numeric = function(con) {
    dbGetQuery(con, "SELECT * FROM benchmark_data WHERE value_int > 5000")
  },
  
  filter_string = function(con) {
    dbGetQuery(con, "SELECT * FROM benchmark_data WHERE category = 'A'")
  },
  
  group_by_sum = function(con) {
    dbGetQuery(con, "
      SELECT category, SUM(value_float) as total_value, COUNT(*) as n
      FROM benchmark_data 
      GROUP BY category
    ")
  },
  
  group_by_multi = function(con) {
    dbGetQuery(con, "
      SELECT category, region, 
             SUM(value_float) as total_value,
             AVG(value_int) as avg_int,
             COUNT(*) as n
      FROM benchmark_data 
      GROUP BY category, region
    ")
  },
  
  summary_stats = function(con) {
    dbGetQuery(con, "
      SELECT 
        AVG(value_int) as mean_int, MIN(value_int) as min_int, MAX(value_int) as max_int,
        AVG(value_float) as mean_float, MIN(value_float) as min_float, MAX(value_float) as max_float,
        AVG(quantity) as mean_qty, MIN(quantity) as min_qty, MAX(quantity) as max_qty
      FROM benchmark_data
    ")
  }
)

# ============================================================
# SQLite Query Functions (same SQL, different connection)
# ============================================================

sqlite_queries <- list(
  full_scan = function(con) {
    dbGetQuery(con, "SELECT * FROM benchmark_data LIMIT 10000")
  },
  
  filter_numeric = function(con) {
    dbGetQuery(con, "SELECT * FROM benchmark_data WHERE value_int > 5000")
  },
  
  filter_string = function(con) {
    dbGetQuery(con, "SELECT * FROM benchmark_data WHERE category = 'A'")
  },
  
  group_by_sum = function(con) {
    dbGetQuery(con, "
      SELECT category, SUM(value_float) as total_value, COUNT(*) as n
      FROM benchmark_data 
      GROUP BY category
    ")
  },
  
  group_by_multi = function(con) {
    dbGetQuery(con, "
      SELECT category, region, 
             SUM(value_float) as total_value,
             AVG(value_int) as avg_int,
             COUNT(*) as n
      FROM benchmark_data 
      GROUP BY category, region
    ")
  },
  
  summary_stats = function(con) {
    dbGetQuery(con, "
      SELECT 
        AVG(value_int) as mean_int, MIN(value_int) as min_int, MAX(value_int) as max_int,
        AVG(value_float) as mean_float, MIN(value_float) as min_float, MAX(value_float) as max_float,
        AVG(quantity) as mean_qty, MIN(quantity) as min_qty, MAX(quantity) as max_qty
      FROM benchmark_data
    ")
  }
)

# ============================================================
# Parquet Query Functions (using DuckDB to query parquet)
# ============================================================

parquet_queries <- list(
  full_scan = function(con, path) {
    dbGetQuery(con, sprintf("SELECT * FROM read_parquet('%s') LIMIT 10000", path))
  },
  
  filter_numeric = function(con, path) {
    dbGetQuery(con, sprintf("SELECT * FROM read_parquet('%s') WHERE value_int > 5000", path))
  },
  
  filter_string = function(con, path) {
    dbGetQuery(con, sprintf("SELECT * FROM read_parquet('%s') WHERE category = 'A'", path))
  },
  
  group_by_sum = function(con, path) {
    dbGetQuery(con, sprintf("
      SELECT category, SUM(value_float) as total_value, COUNT(*) as n
      FROM read_parquet('%s') 
      GROUP BY category
    ", path))
  },
  
  group_by_multi = function(con, path) {
    dbGetQuery(con, sprintf("
      SELECT category, region, 
             SUM(value_float) as total_value,
             AVG(value_int) as avg_int,
             COUNT(*) as n
      FROM read_parquet('%s') 
      GROUP BY category, region
    ", path))
  },
  
  summary_stats = function(con, path) {
    dbGetQuery(con, sprintf("
      SELECT 
        AVG(value_int) as mean_int, MIN(value_int) as min_int, MAX(value_int) as max_int,
        AVG(value_float) as mean_float, MIN(value_float) as min_float, MAX(value_float) as max_float,
        AVG(quantity) as mean_qty, MIN(quantity) as min_qty, MAX(quantity) as max_qty
      FROM read_parquet('%s')
    ", path))
  }
)

# ============================================================
# CSV Query Functions (using data.table fread)
# ============================================================

csv_queries <- list(
  full_scan = function(path) {
    fread(path, nrows = 10000)
  },
  
  filter_numeric = function(path) {
    dt <- fread(path)
    dt[value_int > 5000]
  },
  
  filter_string = function(path) {
    dt <- fread(path)
    dt[category == "A"]
  },
  
  group_by_sum = function(path) {
    dt <- fread(path)
    dt[, .(total_value = sum(value_float), n = .N), by = category]
  },
  
  group_by_multi = function(path) {
    dt <- fread(path)
    dt[, .(total_value = sum(value_float), avg_int = mean(value_int), n = .N), 
       by = .(category, region)]
  },
  
  summary_stats = function(path) {
    dt <- fread(path)
    data.table(
      mean_int = mean(dt$value_int), min_int = min(dt$value_int), max_int = max(dt$value_int),
      mean_float = mean(dt$value_float), min_float = min(dt$value_float), max_float = max(dt$value_float),
      mean_qty = mean(dt$quantity), min_qty = min(dt$quantity), max_qty = max(dt$quantity)
    )
  }
)

cat("Query functions defined for all 4 engines:\n")
cat("  - DuckDB (SQL)\n")
cat("  - SQLite (SQL)\n")
cat("  - Parquet via DuckDB (SQL)\n")
cat("  - CSV via data.table (R native)\n")
cat("\nQuery types:\n")
cat(paste("  -", names(duckdb_queries), collapse = "\n"))

## 5. Run Microbenchmarks

Execute each query type 100 times per engine and collect timing data. Also track peak memory usage during each query type.

In [ ]:
# Benchmark runner function with memory profiling
run_benchmark <- function(query_func, ..., times = N_BENCHMARK_RUNS) {
  # Warm-up run
  result <- query_func(...)
  
  # Timed runs
  timings <- numeric(times)
  for (i in seq_len(times)) {
    gc(verbose = FALSE)  # Clean up before each run
    start <- Sys.time()
    result <- query_func(...)
    timings[i] <- as.numeric(difftime(Sys.time(), start, units = "secs")) * 1000  # ms
  }
  
  return(timings)
}

# Memory tracking function
get_memory_usage <- function(query_func, ...) {
  gc(verbose = FALSE)
  mem_before <- pryr::mem_used()
  result <- query_func(...)
  mem_after <- pryr::mem_used()
  gc(verbose = FALSE)
  return(as.numeric(mem_after - mem_before) / (1024^2))  # MB
}

# Store all results
benchmark_results <- data.frame()
memory_results <- data.frame()

query_types <- names(duckdb_queries)

cat("Running benchmarks...\n")
cat(sprintf("Each query will be run %d times\n\n", N_BENCHMARK_RUNS))

# ============================================================
# DuckDB Benchmarks
# ============================================================
cat("=== DuckDB ===\n")
con_duck <- dbConnect(duckdb(), dbdir = duckdb_path, read_only = TRUE)

for (query_name in query_types) {
  cat(sprintf("  %s...", query_name))
  
  timings <- run_benchmark(duckdb_queries[[query_name]], con_duck)
  mem_used <- get_memory_usage(duckdb_queries[[query_name]], con_duck)
  
  benchmark_results <- rbind(benchmark_results, data.frame(
    engine = "DuckDB",
    query_type = query_name,
    run = seq_along(timings),
    duration_ms = timings
  ))
  
  memory_results <- rbind(memory_results, data.frame(
    engine = "DuckDB",
    query_type = query_name,
    memory_mb = mem_used
  ))
  
  cat(sprintf(" mean=%.2fms\n", mean(timings)))
}
dbDisconnect(con_duck, shutdown = TRUE)

# ============================================================
# SQLite Benchmarks
# ============================================================
cat("\n=== SQLite ===\n")
con_sqlite <- dbConnect(RSQLite::SQLite(), sqlite_path)

for (query_name in query_types) {
  cat(sprintf("  %s...", query_name))
  
  timings <- run_benchmark(sqlite_queries[[query_name]], con_sqlite)
  mem_used <- get_memory_usage(sqlite_queries[[query_name]], con_sqlite)
  
  benchmark_results <- rbind(benchmark_results, data.frame(
    engine = "SQLite",
    query_type = query_name,
    run = seq_along(timings),
    duration_ms = timings
  ))
  
  memory_results <- rbind(memory_results, data.frame(
    engine = "SQLite",
    query_type = query_name,
    memory_mb = mem_used
  ))
  
  cat(sprintf(" mean=%.2fms\n", mean(timings)))
}
dbDisconnect(con_sqlite)

# ============================================================
# Parquet Benchmarks (via DuckDB)
# ============================================================
cat("\n=== Parquet (via DuckDB) ===\n")
con_parquet <- dbConnect(duckdb())

for (query_name in query_types) {
  cat(sprintf("  %s...", query_name))
  
  timings <- run_benchmark(parquet_queries[[query_name]], con_parquet, parquet_path)
  mem_used <- get_memory_usage(parquet_queries[[query_name]], con_parquet, parquet_path)
  
  benchmark_results <- rbind(benchmark_results, data.frame(
    engine = "Parquet",
    query_type = query_name,
    run = seq_along(timings),
    duration_ms = timings
  ))
  
  memory_results <- rbind(memory_results, data.frame(
    engine = "Parquet",
    query_type = query_name,
    memory_mb = mem_used
  ))
  
  cat(sprintf(" mean=%.2fms\n", mean(timings)))
}
dbDisconnect(con_parquet, shutdown = TRUE)

# ============================================================
# CSV Benchmarks (via data.table)
# ============================================================
cat("\n=== CSV (via data.table) ===\n")

for (query_name in query_types) {
  cat(sprintf("  %s...", query_name))
  
  timings <- run_benchmark(csv_queries[[query_name]], csv_path)
  mem_used <- get_memory_usage(csv_queries[[query_name]], csv_path)
  
  benchmark_results <- rbind(benchmark_results, data.frame(
    engine = "CSV",
    query_type = query_name,
    run = seq_along(timings),
    duration_ms = timings
  ))
  
  memory_results <- rbind(memory_results, data.frame(
    engine = "CSV",
    query_type = query_name,
    memory_mb = mem_used
  ))
  
  cat(sprintf(" mean=%.2fms\n", mean(timings)))
}

cat("\n=== Benchmarks Complete ===\n")
cat(sprintf("Total observations: %d\n", nrow(benchmark_results)))

## 6. Compute Summary Statistics

In [ ]:
# Compute summary statistics by engine and query type
benchmark_summary <- benchmark_results %>%
  group_by(engine, query_type) %>%
  summarise(
    mean_ms = mean(duration_ms),
    median_ms = median(duration_ms),
    sd_ms = sd(duration_ms),
    min_ms = min(duration_ms),
    max_ms = max(duration_ms),
    p25_ms = quantile(duration_ms, 0.25),
    p75_ms = quantile(duration_ms, 0.75),
    .groups = "drop"
  ) %>%
  arrange(query_type, mean_ms)

# Add speedup relative to CSV baseline
benchmark_summary <- benchmark_summary %>%
  group_by(query_type) %>%
  mutate(
    csv_baseline = mean_ms[engine == "CSV"],
    speedup_vs_csv = csv_baseline / mean_ms
  ) %>%
  ungroup()

# Display summary table
cat("=== Benchmark Summary (milliseconds) ===\n\n")
print(as.data.frame(benchmark_summary), digits = 2)

# Pivot for easier comparison
comparison_table <- benchmark_summary %>%
  select(engine, query_type, mean_ms, speedup_vs_csv) %>%
  pivot_wider(
    names_from = engine,
    values_from = c(mean_ms, speedup_vs_csv),
    names_glue = "{engine}_{.value}"
  )

cat("\n=== Mean Query Time by Engine (ms) ===\n")
mean_pivot <- benchmark_summary %>%
  select(engine, query_type, mean_ms) %>%
  pivot_wider(names_from = engine, values_from = mean_ms)
print(as.data.frame(mean_pivot), digits = 2)

cat("\n=== Speedup vs CSV Baseline ===\n")
speedup_pivot <- benchmark_summary %>%
  select(engine, query_type, speedup_vs_csv) %>%
  pivot_wider(names_from = engine, values_from = speedup_vs_csv)
print(as.data.frame(speedup_pivot), digits = 2)

## 7. Visualize Results

### 7.1 Mean Query Time Comparison (Bar Chart)

In [ ]:
# Set theme
theme_set(theme_minimal(base_size = 12))
engine_colors <- c("DuckDB" = "#FFA500", "SQLite" = "#4169E1", 
                   "Parquet" = "#32CD32", "CSV" = "#DC143C")

# Bar chart: Mean query time by engine and query type
p1 <- ggplot(benchmark_summary, 
             aes(x = reorder(query_type, mean_ms), y = mean_ms, fill = engine)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.7) +
  geom_errorbar(aes(ymin = mean_ms - sd_ms, ymax = mean_ms + sd_ms),
                position = position_dodge(width = 0.8), width = 0.25, alpha = 0.7) +
  scale_fill_manual(values = engine_colors) +
  coord_flip() +
  labs(
    title = "Mean Query Time by Engine and Query Type",
    subtitle = sprintf("Error bars show ±1 SD across %d runs", N_BENCHMARK_RUNS),
    x = "Query Type",
    y = "Time (milliseconds)",
    fill = "Engine"
  ) +
  theme(
    legend.position = "bottom",
    plot.title = element_text(face = "bold", size = 14),
    panel.grid.major.y = element_blank()
  )

print(p1)

### 7.2 Distribution of Query Times (Box Plots)

In [ ]:
# Box plots: Distribution of query times
p2 <- ggplot(benchmark_results, 
             aes(x = engine, y = duration_ms, fill = engine)) +
  geom_boxplot(outlier.alpha = 0.3) +
  facet_wrap(~ query_type, scales = "free_y", ncol = 3) +
  scale_fill_manual(values = engine_colors) +
  labs(
    title = "Distribution of Query Times by Engine",
    subtitle = sprintf("Based on %d runs per query type", N_BENCHMARK_RUNS),
    x = "Engine",
    y = "Time (milliseconds)"
  ) +
  theme(
    legend.position = "none",
    plot.title = element_text(face = "bold", size = 14),
    axis.text.x = element_text(angle = 45, hjust = 1),
    strip.text = element_text(face = "bold")
  )

print(p2)

### 7.3 Speedup Heatmap (Relative to CSV Baseline)

In [ ]:
# Heatmap: Speedup relative to CSV
heatmap_data <- benchmark_summary %>%
  select(engine, query_type, speedup_vs_csv) %>%
  mutate(
    speedup_label = sprintf("%.1fx", speedup_vs_csv),
    engine = factor(engine, levels = c("CSV", "SQLite", "Parquet", "DuckDB"))
  )

p3 <- ggplot(heatmap_data, 
             aes(x = engine, y = query_type, fill = speedup_vs_csv)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = speedup_label), color = "black", size = 4, fontface = "bold") +
  scale_fill_gradient2(
    low = "#DC143C", mid = "#FFFF99", high = "#32CD32",
    midpoint = 1, 
    limits = c(0, max(heatmap_data$speedup_vs_csv) * 1.1),
    name = "Speedup\nvs CSV"
  ) +
  labs(
    title = "Speedup Relative to CSV Baseline",
    subtitle = "Green = faster than CSV, Red = slower than CSV",
    x = "Engine",
    y = "Query Type"
  ) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    axis.text = element_text(size = 11),
    panel.grid = element_blank(),
    panel.background = element_rect(fill = "white")
  )

print(p3)

### 7.4 File Size Comparison

In [ ]:
# File size comparison
file_sizes <- data.frame(
  Engine = c("DuckDB", "SQLite", "Parquet", "CSV"),
  Size_MB = c(write_stats$duckdb$size, write_stats$sqlite$size,
              write_stats$parquet$size, write_stats$csv$size) / (1024^2),
  Write_Time_sec = c(write_stats$duckdb$time, write_stats$sqlite$time,
                     write_stats$parquet$time, write_stats$csv$time)
)

# Calculate compression ratio relative to CSV
file_sizes$Compression_Ratio <- file_sizes$Size_MB[file_sizes$Engine == "CSV"] / file_sizes$Size_MB

p4 <- ggplot(file_sizes, aes(x = reorder(Engine, -Size_MB), y = Size_MB, fill = Engine)) +
  geom_bar(stat = "identity", width = 0.6) +
  geom_text(aes(label = sprintf("%.1f MB\n(%.1fx)", Size_MB, Compression_Ratio)), 
            vjust = -0.3, size = 3.5, fontface = "bold") +
  scale_fill_manual(values = engine_colors) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  labs(
    title = "File Size Comparison",
    subtitle = "Compression ratio relative to CSV shown in parentheses",
    x = "Engine",
    y = "File Size (MB)"
  ) +
  theme(
    legend.position = "none",
    plot.title = element_text(face = "bold", size = 14)
  )

print(p4)

# Print summary table
cat("\n=== File Size Summary ===\n")
print(file_sizes)

### 7.5 Memory Usage Comparison

In [ ]:
# Memory usage comparison
p5 <- ggplot(memory_results, 
             aes(x = query_type, y = memory_mb, fill = engine)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.7) +
  scale_fill_manual(values = engine_colors) +
  coord_flip() +
  labs(
    title = "Peak Memory Usage by Query Type",
    subtitle = "Memory allocated during query execution",
    x = "Query Type",
    y = "Memory (MB)",
    fill = "Engine"
  ) +
  theme(
    legend.position = "bottom",
    plot.title = element_text(face = "bold", size = 14),
    panel.grid.major.y = element_blank()
  )

print(p5)

# Memory summary table
cat("\n=== Memory Usage Summary (MB) ===\n")
memory_pivot <- memory_results %>%
  pivot_wider(names_from = engine, values_from = memory_mb)
print(as.data.frame(memory_pivot), digits = 2)

### 7.6 Combined Dashboard

In [ ]:
# Combined dashboard using patchwork (if available) or gridExtra
if (!requireNamespace("patchwork", quietly = TRUE)) {
  install.packages("patchwork")
}
library(patchwork)

# Create combined dashboard
dashboard <- (p1 + p4) / (p3 + p5) +
  plot_annotation(
    title = "Flat File Database Benchmark Results",
    subtitle = sprintf("1M rows × 8 columns | %d benchmark runs per query", N_BENCHMARK_RUNS),
    theme = theme(
      plot.title = element_text(face = "bold", size = 18, hjust = 0.5),
      plot.subtitle = element_text(size = 12, hjust = 0.5)
    )
  )

# Display dashboard (larger size)
options(repr.plot.width = 16, repr.plot.height = 14)
print(dashboard)

## 8. Save Results and Cleanup

In [ ]:
# Save benchmark results to CSV
results_dir <- "benchmark_results"
if (!dir.exists(results_dir)) {
  dir.create(results_dir)
}

# Save raw benchmark data
fwrite(as.data.table(benchmark_results), 
       file.path(results_dir, "benchmark_raw_results.csv"))

# Save summary statistics
fwrite(as.data.table(benchmark_summary), 
       file.path(results_dir, "benchmark_summary.csv"))

# Save file size comparison
fwrite(as.data.table(file_sizes), 
       file.path(results_dir, "file_sizes.csv"))

# Save memory usage
fwrite(as.data.table(memory_results), 
       file.path(results_dir, "memory_usage.csv"))

# Save plots
ggsave(file.path(results_dir, "query_time_comparison.png"), p1, width = 12, height = 8, dpi = 150)
ggsave(file.path(results_dir, "query_time_boxplots.png"), p2, width = 14, height = 10, dpi = 150)
ggsave(file.path(results_dir, "speedup_heatmap.png"), p3, width = 10, height = 8, dpi = 150)
ggsave(file.path(results_dir, "file_sizes.png"), p4, width = 10, height = 6, dpi = 150)
ggsave(file.path(results_dir, "memory_usage.png"), p5, width = 12, height = 8, dpi = 150)
ggsave(file.path(results_dir, "benchmark_dashboard.png"), dashboard, width = 16, height = 14, dpi = 150)

cat("=== Results saved to:", results_dir, "===\n")
cat("\nFiles created:\n")
list.files(results_dir)

## 9. Conclusions

In [ ]:
# Generate conclusions based on results
cat("=== BENCHMARK CONCLUSIONS ===\n\n")

# Find fastest engine per query type
fastest_per_query <- benchmark_summary %>%
  group_by(query_type) %>%
  slice_min(mean_ms, n = 1) %>%
  select(query_type, engine, mean_ms)

cat("FASTEST ENGINE PER QUERY TYPE:\n")
for (i in seq_len(nrow(fastest_per_query))) {
  cat(sprintf("  • %s: %s (%.2f ms)\n", 
              fastest_per_query$query_type[i],
              fastest_per_query$engine[i],
              fastest_per_query$mean_ms[i]))
}

# Overall rankings
overall_ranking <- benchmark_summary %>%
  group_by(engine) %>%
  summarise(
    mean_time_ms = mean(mean_ms),
    median_time_ms = median(mean_ms),
    .groups = "drop"
  ) %>%
  arrange(mean_time_ms)

cat("\nOVERALL PERFORMANCE RANKING (by mean query time):\n")
for (i in seq_len(nrow(overall_ranking))) {
  cat(sprintf("  %d. %s: %.2f ms average\n", 
              i, overall_ranking$engine[i], overall_ranking$mean_time_ms[i]))
}

# File size ranking
size_ranking <- file_sizes %>% arrange(Size_MB)
cat("\nFILE SIZE RANKING (smallest to largest):\n")
for (i in seq_len(nrow(size_ranking))) {
  cat(sprintf("  %d. %s: %.1f MB (%.1fx compression vs CSV)\n", 
              i, size_ranking$Engine[i], size_ranking$Size_MB[i], size_ranking$Compression_Ratio[i]))
}

# Key takeaways
cat("\n=== KEY TAKEAWAYS ===\n")
cat("
1. QUERY PERFORMANCE:
   - DuckDB and Parquet excel at analytical queries (GROUP BY, aggregations)
   - SQLite is competitive for simple queries but slower for complex analytics
   - CSV requires full file read for every operation (no query optimization)

2. FILE SIZE:
   - Parquet offers best compression (columnar format + compression)
   - DuckDB has moderate file size with embedded indexes
   - CSV is largest due to no compression

3. MEMORY USAGE:
   - DuckDB and Parquet are more memory-efficient for large datasets
   - CSV loads entire file into memory for each query

4. RECOMMENDATIONS:
   - For analytical workloads: DuckDB or Parquet
   - For simple key-value lookups: SQLite
   - For data interchange: Parquet (portable, compressed)
   - Avoid CSV for large datasets and repeated queries
")

cat("\n=== SESSION INFO ===\n")
sessionInfo()